# Joint comparison of candidate tilt mechanisms

This final notebook is run after the component notebooks pass QC. It asks which mechanisms retain within-eddy and between-eddy associations after adjustment. Direction and magnitude are analysed separately, AE and CE are kept separate, and whole eddies remain the clustering unit.

The goal is not to declare causality from a single coefficient. A mechanism is considered supported only when its directional, magnitude, timescale, regime and within-eddy predictions agree.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


Rows: 127,426; measured tilts: 105,621; eddies: 2,982


In [2]:
from beta_effect_background_flow.background_flow_tools import BackgroundConfig, load_background_cache

N2_CACHE = Path("/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day_v2.parquet")
background = load_background_cache(BackgroundConfig()).drop(columns=["ic", "jc", "month"], errors="ignore")
n2 = pd.read_parquet(N2_CACHE)

data = mech.merge_one_to_one_or_many_to_one(df, background)
data = mech.merge_one_to_one_or_many_to_one(data, n2)
data = tilt.add_pv_gradient_terms(data, grid)
data = mech.add_stratification_proxies(data)
data = mech.add_background_shear(data)
data = mech.add_accumulated_shear(data, "ann_500", windows=(10, 20, 30))
data = tilt.add_region_labels(data, grid)
data = mech.add_topographic_regimes(data, shelf_depth=2000.0, dominance_ratio=2.0)
data["slope_mag"] = np.hypot(data.dhdx, data.dhdy)


FileNotFoundError: [Errno 2] No such file or directory: '/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day.parquet'

In [ ]:
# Predeclared mechanism variables. Add wind only after its cache passes QC.
mechanisms = [
    "beta", "N2_500m_s2", "N2_over_f2_500m", "Bu_proxy_500m",
    "ann_500_shear_mag_ms", "ann_500_accum_20d_mag_km",
    "slope_mag", "topo_plan_ratio_raw", "Rc", "h",
]
varying = ["ann_500_shear_mag_ms", "ann_500_accum_20d_mag_km", "Rc"]
data = mech.standardise_within_between(data, varying)
display(data[mechanisms].describe().T)


In [ ]:
# Magnitude: compare standardized clustered models, then inspect effect sizes and CIs.
import statsmodels.formula.api as smf

model_sets = {
    "environment": ["beta", "N2_over_f2_500m", "Rc", "h"],
    "plus_shear": ["beta", "N2_over_f2_500m", "Rc", "h",
                   "ann_500_accum_20d_mag_km_between", "ann_500_accum_20d_mag_km_within"],
    "plus_topography": ["beta", "N2_over_f2_500m", "Rc", "h", "slope_mag", "topo_plan_ratio_raw",
                        "ann_500_accum_20d_mag_km_between", "ann_500_accum_20d_mag_km_within"],
}
fits = {}
for cyc, part in data.groupby("Cyc"):
    for name, columns in model_sets.items():
        use = part[["Eddy", "TiltDis", *columns]].replace([np.inf, -np.inf], np.nan).dropna().copy()
        for column in columns:
            sd = use[column].std()
            use[f"z_{column}"] = (use[column] - use[column].mean()) / sd if sd > 0 else 0
        formula = "np.log1p(TiltDis) ~ " + " + ".join(f"z_{c}" for c in columns)
        fits[(cyc, name)] = smf.gee(formula, groups="Eddy", data=use).fit()
        print(cyc, name, "rows", len(use), "eddies", use.Eddy.nunique())


In [ ]:
for key, fit in fits.items():
    print("\n", key)
    display(pd.DataFrame({"estimate": fit.params, "ci_low": fit.conf_int()[0], "ci_high": fit.conf_int()[1]}))


In [ ]:
# Directional scorecard assembled from the component notebooks.
direction_metrics = [
    "ann_500_tilt_shear_offset",
    "ann_500_accum_20d_offset",
]
data["tilt_planetary_pv_offset"] = mech.signed_angle_difference(data.TiltDir, data.PV_grad_plan_theta)
data["tilt_topographic_pv_offset"] = mech.signed_angle_difference(data.TiltDir, data.PV_grad_topo_theta)
direction_metrics += ["tilt_planetary_pv_offset", "tilt_topographic_pv_offset"]
scorecard = mech.circular_offset_summary(data, direction_metrics, group=("Cyc", "ShelfRegime", "PVRegime"))
display(scorecard.sort_values(["Cyc", "ShelfRegime", "resultant_length"], ascending=[True, True, False]))


## Final evidence rubric

For each mechanism report:

1. **Direction:** Is the eddy-equal offset concentrated around the predicted direction?
2. **Magnitude:** Does forcing magnitude predict `TiltDis` with a scientifically meaningful effect?
3. **Timescale:** Does a plausible trailing window outperform instantaneous forcing?
4. **Within eddies:** Does the same eddy respond when forcing changes?
5. **Regime:** Does the relationship strengthen where the mechanism should dominate?
6. **Robustness:** Does it survive spatial blocks, tilt thresholds, depth choice and background definition?

Use language such as “supports,” “is consistent with,” or “does not support.” Reserve causal language for a coherent suite of predictions, not isolated significance.
